In [1]:
%pip install numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import numpy as np
import pandas as pd

What variable we want to explore the correlation with single or multi listing

estimated_revenue_l365d
estimated_occupancy_l365d
number_of_reviews_ly
price
...

In [3]:
listings_raw = pd.read_csv('listings_raw.csv')
reviews_raw = pd.read_csv('reviews_raw.csv')

listings_raw.head(5)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,10803,https://www.airbnb.com/rooms/10803,20260616211523,2026-06-17,city scrape,"Room in Deco Apartment, Brunswick East",A large air conditioned room with firm queen s...,NaN,https://a0.muscache.com/pictures/e5f30dd1-ac57...,38901,...,4.73,4.70,4.66,NaN,NaN,1,0,1,0,1.31
1,12936,https://www.airbnb.com/rooms/12936,20260616211523,2026-06-28,previous scrape,St Kilda 1BR+BEACHSIDE+BALCONY+WIFI+AC,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,NaN,https://a0.muscache.com/pictures/59701/2e8cdaf...,50121,...,4.83,4.78,4.66,NaN,NaN,10,10,0,0,0.22
2,41836,https://www.airbnb.com/rooms/41836,20260616211523,2026-06-28,previous scrape,CLOSE TO CITY & MELBOURNE AIRPORT,Easy to travel from and to the Airport; quiet ...,NaN,https://a0.muscache.com/pictures/569696dd-1ad0...,182833,...,4.83,4.39,4.69,NaN,NaN,2,0,2,0,0.83
3,43429,https://www.airbnb.com/rooms/43429,20260616211523,2026-06-17,city scrape,Tranquil Javanese Studio and Pond!,"No service/Cleaning Fees, EV Charger, Study th...",NaN,https://a0.muscache.com/pictures/airflow/Hosti...,189684,...,4.94,4.79,4.86,NaN,NaN,2,2,0,0,1.48
4,44699,https://www.airbnb.com/rooms/44699,20260616211523,2026-06-17,city scrape,"15 yearsHosting (4.8), 8 CITY TRAMS, GymPoolTe...",Unwavering service — just ask and we do our be...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,189245,...,4.97,4.84,4.71,NaN,NaN,1,0,1,0,0.35


In [4]:
print(listings_raw.shape)
print(reviews_raw.shape)

(25728, 90)
(1026690, 6)


clean price into int for comparison

In [5]:
def parse_price(df):
    df = df.copy()
    
    def parse_price_int(price):
        """
        Independently re-implement the parsing logic in a different way 
        """
        if pd.isna(price):
            return np.nan

        clean_price = str(price).replace("$", "").replace(",", "").strip()
        try:
            return float(clean_price)
        except:
            return np.nan

    df['price_int'] = df['price'].apply(parse_price_int)

    return df

turn the number of listings into only single or multilisting

In [6]:
def listing_class(df):
    df = df.copy()
    
    def listing_class_logic(num_listings):

        if pd.isna(num_listings):
            return np.nan
        else:
            num_listings = int(num_listings)
        
        if num_listings <= 0:
            return np.nan
        elif num_listings == 1:
            return "Singlelisting"
        else:
            return "Multilisting"

    df['listing_class'] = df['calculated_host_listings_count'].apply(listing_class_logic)

    return df

# EDA

## Before Preprocessing

In [7]:
def column_audit(df):
    rows = len(df)

    result = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_missing": df.notna().sum(),
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean().mul(100),
        "unique_count": df.nunique(dropna=True)
    })

    result["unique_percent"] = (
        result["unique_count"] / rows * 100
    )

    return result.sort_values(
        ["missing_percent", "unique_count"],
        ascending=[False, True]
    )

In [8]:
listings_audit = column_audit(listings_raw)
reviews_audit = column_audit(reviews_raw)

display(listings_audit)
display(reviews_audit)

,dtype,non_missing,missing_count,missing_percent,unique_count,unique_percent
neighborhood_overview,float64,0,25728,100.0,0,0.000000
host_since,float64,0,25728,100.0,0,0.000000
host_response_time,float64,0,25728,100.0,0,0.000000
host_response_rate,float64,0,25728,100.0,0,0.000000
host_acceptance_rate,float64,0,25728,100.0,0,0.000000
...,...,...,...,...,...,...
amenities,str,25728,0,0.0,24395,94.818874
name,str,25728,0,0.0,25021,97.252021
picture_url,str,25728,0,0.0,25196,97.932214
id,int64,25728,0,0.0,25728,100.000000


,dtype,non_missing,missing_count,missing_percent,unique_count,unique_percent
comments,str,1026557,133,0.012954,979476,95.401338
reviewer_name,str,1026689,1,0.000097,121850,11.868237
date,str,1026690,0,0.000000,5197,0.506190
listing_id,int64,1026690,0,0.000000,21251,2.069856
reviewer_id,int64,1026690,0,0.000000,794203,77.355677
id,int64,1026690,0,0.000000,1026690,100.000000


In [9]:
listings_all_missing =  listings_raw.columns[
    listings_raw.isna().all()
].tolist()

reviews_all_missing = reviews_raw.columns[
    reviews_raw.isna().all()
].tolist()

print("Listings columns with 100% missing:")
print(listings_all_missing)

print("Reviews columns with 100% missing:")
print(reviews_all_missing)

Listings columns with 100% missing:
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated', 'license', 'instant_bookable']
Reviews columns with 100% missing:
[]


In [10]:
listings_work = listings_raw.drop(
    columns=listings_all_missing
).copy()

reviews_work = reviews_raw.copy()

print("Listings before:", listings_raw.shape)
print("Listings after:", listings_work.shape)

print("Reviews before:", reviews_raw.shape)
print("Reviews after:", reviews_work.shape)

Listings before: (25728, 90)
Listings after: (25728, 76)
Reviews before: (1026690, 6)
Reviews after: (1026690, 6)


# Property type preprocessing

In [11]:
counts = listings_work['property_type'].value_counts()
rare_types = counts[counts < 10]

print(f"Number of rare categories (< 10 listings): {len(rare_types)}")
print(rare_types)

Number of rare categories (< 10 listings): 41
property_type
Private room in casa particular     9
Private room in tiny home           7
Shared room in guesthouse           7
Private room in cottage             6
Shared room in hostel               6
Earthen home                        5
Private room in earthen home        4
Entire chalet                       4
Yurt                                4
Private room in nature lodge        4
Private room in vacation home       4
Private room in kezhan              4
Treehouse                           3
Castle                              3
Room in nature lodge                3
Shared room in condo                3
Hut                                 3
Private room in cabin               3
Casa particular                     2
Tower                               2
Private room in treehouse           2
Tent                                2
Bus                                 2
Shared room in townhouse            2
Entire bed and breakfast    

In [12]:
df = listings_work
print(df["property_type"].value_counts().to_string())

property_type
Entire rental unit                    11262
Entire home                            4512
Private room in home                   3494
Private room in rental unit            1742
Entire condo                            643
Entire townhouse                        522
Entire serviced apartment               434
Entire guesthouse                       405
Private room in townhouse               372
Entire guest suite                      253
Room in hotel                           202
Shared room in home                     168
Private room in condo                   164
Entire cottage                          161
Room in aparthotel                      150
Entire loft                             114
Entire cabin                             91
Private room in villa                    89
Farm stay                                85
Tiny home                                84
Private room in bed and breakfast        72
Entire villa                             63
Private room in gu

In [13]:
df["property_type_grouped"] = "Other"

df.loc[df["property_type"].str.contains(
      "rental unit", case=False, na= False
), "property_type_grouped"

] = "Rental unit"

df.loc[df["property_type"].str.contains(
      "home|townhouse|guesthouse|cabin|villa|cottage", case=False, na=False
), "property_type_grouped"

] = "House"

df.loc[df["property_type"].str.contains(
    "apartment|condo|loft", case=False, na=False
), "property_type_grouped"

] = "Apartment"

df.loc[df["property_type"].str.contains(
    "hotel|hostel|bed and breakfast", case=False, na=False
), "property_type_grouped"

] = "Hotel/Hostel/B&B"

In [14]:
df[["property_type", "property_type_grouped"]]

,property_type,property_type_grouped
0,Private room in rental unit,Rental unit
1,Entire rental unit,Rental unit
2,Private room in home,House
3,Entire rental unit,Rental unit
4,Private room in condo,Apartment
...,...,...
25723,Private room in rental unit,Rental unit
25724,Entire rental unit,Rental unit
25725,Entire rental unit,Rental unit
25726,Entire rental unit,Rental unit


# Accomodates Preprocess

discretisation of the data

In [15]:
df["accommodates"].describe()

count    25728.000000
mean         3.767996
std          2.415809
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         16.000000
Name: accommodates, dtype: float64

positively skewed distribution; mode < median < mean

In [16]:
df["accommodates"].value_counts().sort_index()

accommodates
1     1985
2     9447
3     1452
4     6115
5     1290
6     2849
7      410
8     1127
9      190
10     420
11      70
12     152
13      26
14      58
15      28
16     109
Name: count, dtype: int64

Separate the number of accomodates into bins ("1-2", "2-3", "3-4", "5-6", "6-7", "7+")

In [17]:
max_val = int(df["accommodates"].max())

bin_vals = [0, 2, 4, 6, max_val + 1]

bin_labels = ["1-2", "3-4", "5-6", "7+"]

df["accommodate_bins"] = pd.cut(df["accommodates"], bins=bin_vals, 
                                   labels=bin_labels, include_lowest=True)

In [24]:
accommodates_counts = df["accommodate_bins"].value_counts().sort_index()
accommodates_summary = pd.DataFrame({
    "Accommodates Range": accommodates_counts.index,
    "Number of Listings": accommodates_counts.values
})
print(accommodates_summary)

  Accommodates Range  Number of Listings
0                1-2               11432
1                3-4                7567
2                5-6                4139
3                 7+                2590


# Room type Preprocess

use one hot encoding matrix

we need to use one-hot encoding because room_Type has no numerical order

In [26]:
room_type_counts = df["room_type"].value_counts()
room_type_pct = room_type_counts / len(df) * 100

room_type_summary = pd.DataFrame({
    "Count" : room_type_counts,
    "Percentage(%)" : room_type_pct.round(2)
})

room_type_summary

,Count,Percentage(%)
room_type,,
Entire home/apt,18829,73.18
Private room,6616,25.72
Shared room,231,0.90
Hotel room,52,0.20


The proportion of "Shared" and Hotel" is very small and insignificant, so we combine them to "Other"

In [ ]:
df["room_type_group"] = np.where(
    df["room_type"].isin(["Shared room", "Hotel room"]),
    "Other",
    df["room_type"]
)


0           Private room
1        Entire home/apt
2           Private room
3        Entire home/apt
4           Private room
              ...       
25723       Private room
25724    Entire home/apt
25725    Entire home/apt
25726    Entire home/apt
25727    Entire home/apt
Name: room_type_group, Length: 25728, dtype: str

In [27]:
room_type_group_counts = df["room_type_group"].value_counts()
room_type_group_counts

room_type_group
Entire home/apt    18829
Private room        6616
Other                283
Name: count, dtype: int64

In [33]:
room_type_encoded = pd.get_dummies(
    df["room_type_group"], dtype = int
)
room_type_encoded = room_type_encoded[ ["Entire home/apt", "Private room", "Other"] ]
room_type_encoded


,Entire home/apt,Private room,Other
0,0,1,0
1,1,0,0
2,0,1,0
3,1,0,0
4,0,1,0
...,...,...,...
25723,0,1,0
25724,1,0,0
25725,1,0,0
25726,1,0,0


# host_listings_count Preprocess

In [ ]:
invalid_host_count = (df["host_listings_count"] < 1).sum()
missing_host_count = df["host_listings_count"].isna().sum()

In [36]:
df = df[df["host_listings_count"] >= 1]
df = df[df["host_listings_count"].notna()]
df["host_category"] = np.where(df["host_listings_count"] == 1, 
                      "Single_listing",
                      "Multi-listing"
                      )
df[["host_listings_count", "host_category"]]

,host_listings_count,host_category
0,1,Single_listing
1,10,Multi-listing
2,3,Multi-listing
3,4,Multi-listing
4,3,Multi-listing
...,...,...
25723,1,Single_listing
25724,1,Single_listing
25725,3,Multi-listing
25726,1,Single_listing


In [37]:
host_category_counts = df["host_category"].value_counts()
host_category_counts

host_category
Multi-listing     15252
Single_listing    10354
Name: count, dtype: int64

calculate percentages

In [38]:
host_category_pct = host_category_counts/ len(df) * 100
host_category_pct

host_category
Multi-listing     59.564165
Single_listing    40.435835
Name: count, dtype: float64

check for invalid or missing values

In [39]:
print((df["host_listings_count"] < 1).sum())
print(df["host_listings_count"].isna().sum())

0
0


create a Contigency Table to check relationship with superhost status

In [47]:
contigency_table = pd.crosstab(  
         df["host_category"],
         df["host_is_superhost"]
    )

contigency_table.columns = ["Not Superhost", "Superhost"]
contigency_table.index.name = "Host Category"

contigency_table["Total Listings"] = contigency_table.sum(axis = "columns")

contigency_table["Superhost_proportion (%)"] = ( contigency_table["Superhost"] /
                                       contigency_table["Total Listings"] * 100 ).round(2)

contigency_table

,Not Superhost,Superhost,Total Listings,Superhost_proportion (%)
Host Category,,,,
Multi-listing,9683,5569,15252,36.51
Single_listing,8038,2316,10354,22.37
